# 췌장/다중 장기 CT 분할 (Synapse) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 복부 다중 장기 (Multi-organ) CT 세그멘테이션
- **모달리티**: CT (복부 축방향 슬라이스)
- **태스크**: 9-class segmentation — 배경(0) + 8개 장기
  - 클래스: aorta, gallbladder, spleen, left kidney, right kidney, liver, stomach, pancreas
- **핵심 도전**: 췌장 등 소수 장기의 극심한 불균형, 장기 간 경계 모호, 복잡한 3D 구조

## 2. 모델
- **아키텍처**: TransUNet (ResNet50 + ViT-B/16 Transformer)
- **사전학습**: ImageNet (ResNet50) + ImageNet-21k (ViT-B/16)
- **선택 이유**: Transformer의 장거리 의존성 모델링이 다중 장기 관계 파악에 유리
- **출력**: 9채널 softmax — 다중 클래스 직접 출력
- **주의**: `from networks.vit_seg_modeling import VisionTransformer` (상대 import 금지)

## 3. 데이터셋
- **이름**: Synapse Multi-organ CT (BTCV / Synapse benchmark)
- **규모**: 30 볼륨 — 공식 18 train / 12 test (.npz 슬라이스 + .h5 볼륨)
- **클래스 불균형**: 췌장 등 소수 장기 극심한 불균형 (전체 볼륨 대비 매우 작은 비율)
- **공식 분할**: 공식 train/test 분리 (test .h5 볼륨으로 최종 평가)

## 4. 데이터 준비 (협업자용)
> Google Drive에 zip 파일 업로드 후 Cell 0 실행. 자동 압축 해제.

**취득 방법**:
- Synapse Platform: https://www.synapse.org/#!Synapse:syn3193805/wiki/ (계정 등록 필요)
- train_npz/ 와 test_vol_h5/ 를 통째로 zip 압축

**Google Drive 업로드 경로**:
```
MyDrive/imbalanced-data-LWCE/synapse/
  synapse.zip   ← train_npz/ + test_vol_h5/ 전체 압축
    train_npz/case00XX_sliceYYY.npz   ← 학습용 2D 슬라이스
    test_vol_h5/case00XX.npy.h5       ← 테스트용 3D 볼륨
```

## 5. 전처리 및 도메인 특이점
- 슬라이스 단위 학습, 볼륨 단위 평가 (test .h5에서 슬라이스 예측 → 볼륨 집계)
- 슬라이스 캐시 `/tmp/synapse_slices/` 에 저장 (Drive I/O 병목 방지)
- HU 클리핑 후 [0,1] 정규화, 3채널 복제 (grayscale → pseudo-RGB)
- 평가 지표: 클래스별 Dice + mDice (background 제외 8개 장기 평균)

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 30 |
| `pwce_dice` | alpha | 0.2 ~ 2.5 | 30 |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 60 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | mDice (%) | HD95 (mm) | 출처 |
|------|-----------|-----------|------|
| DS-UNETR++ (2025) | **87.75** | **6.67** | arXiv |
| DIN (2025) | 85.49 | 10.74 | Medical Image Analysis |
| SwinUNet (2021) | 79.13 | 21.55 | ECCV'22 |
| TransUNet (2021, baseline) | 77.48 | 31.69 | arXiv |
| U-Net baseline | ~68~74 | ~39 | 복수 논문 |

> 본 연구 목표: TransUNet baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: 클래스별 Dice, mDice (background 제외)
> 결과 저장: `medical_data/results/Pancreas_MultiOrgan_CT/`

# 췌장/다중 장기 CT (Synapse) 불균형 세그멘테이션 실험
## 개요
- **모달리티**: CT (Multi-organ)
- **모델**: TransUNet (R50+ViT-B/16)
- **데이터셋**: Synapse Multi-organ CT — 18 train + 12 test volumes
- **태스크**: 9-class segmentation — 배경 + 8개 장기 (aorta, gallbladder, spleen, left kidney, right kidney, liver, stomach, pancreas)
- **주요 불균형**: 췌장 등 소수 장기 극심한 불균형
- **실험 손실 함수**: `ce_dice`, `plwce_dice`, `pwce_dice`, `plwce_focal_dice`

## 데이터 준비
- **Google Drive 업로드**: `MyDrive/imbalanced-data-LWCE/synapse/synapse.zip` (train_npz/ + test_vol_h5/ 압축)
- **공식 다운로드**: https://www.synapse.org/#!Synapse:syn3193805/wiki/ (계정 등록 필요)
- **TransUNet 가중치**: ViT-B/16 pretrained weights 자동 다운로드 (Cell 2 실행 시)

In [10]:
# === Cell 0: 환경 설정 ===
import os, sys, urllib.request, warnings, subprocess
warnings.filterwarnings('ignore')

for pkg in ['optuna', 'albumentations', 'gdown', 'openpyxl', 'h5py', 'ml-collections']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import glob, random, h5py, json
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- custom_losses 경로 (로컬: repo 기준, Colab: Drive에서 복사 후 /tmp) ---
_CL_LOCAL = '/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB = '/tmp/custom_losses'
if os.path.exists(_CL_LOCAL):
    sys.path.insert(0, _CL_LOCAL)
else:
    os.makedirs(_CL_COLAB, exist_ok=True)
    _cl_src = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if os.path.exists(_cl_src):
        import shutil; shutil.copy(_cl_src, _CL_COLAB)
    sys.path.insert(0, _CL_COLAB)
from custom_losses import get_loss_function, calculate_weights

# --- Google Drive 마운트 (Colab) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님 — 로컬/수동 경로 사용')

# --- 결과 저장 경로 ---
if os.path.exists('/root/imbalanced-data-LWCE'):
    RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/Pancreas_MultiOrgan_CT'
else:
    RESULTS_DIR = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/results/Pancreas_MultiOrgan_CT'
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- TransUNet 클론 및 ViT 가중치 ---
TRANSUNET_DIR = '/tmp/TransUNet'
if not os.path.exists(TRANSUNET_DIR):
    print('TransUNet 클론 중...')
    os.system(f'git clone https://github.com/Beckschen/TransUNet.git {TRANSUNET_DIR}')

if TRANSUNET_DIR not in sys.path:
    sys.path.insert(0, TRANSUNET_DIR)

PRETRAINED_DIR = os.path.join(TRANSUNET_DIR, 'model/vit_checkpoint/imagenet21k')
os.makedirs(PRETRAINED_DIR, exist_ok=True)
VIT_WEIGHTS = os.path.join(PRETRAINED_DIR, 'R50+ViT-B_16.npz')
if not os.path.exists(VIT_WEIGHTS):
    print('ViT-R50+B/16 가중치 다운로드 중...')
    url = 'https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz'
    urllib.request.urlretrieve(url, VIT_WEIGHTS)
    print('다운로드 완료')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 마운트 완료
Device: cuda
환경 설정 완료


In [11]:
# === Cell 1: 데이터 로드 ===
#
# [Google Drive 업로드 방법]
#   synapse.zip 파일 하나만 올리면 됩니다:
#   MyDrive/imbalanced-data-LWCE/synapse/synapse.zip
#
#   zip 내부 구조:
#     synapse/train_npz/*.npz
#     synapse/test_vol_h5/*.npy.h5
#
# [데이터 출처]
#   https://drive.google.com/drive/folders/1ACJEoTp-uqfFJ73qS3eUObQh52nGuzCd
# --- 구글 드라이브 zip 경로 설정 (여기만 수정) ---
GDRIVE_ZIP = '/content/drive/MyDrive/imbalanced-data-LWCE/synapse/synapse.zip'

# --- 로컬 캐시 경로 (/tmp) ---
DATA_DIR      = '/tmp/synapse_data'
TRAIN_NPZ_DIR = os.path.join(DATA_DIR, 'train_npz')
TEST_H5_DIR   = os.path.join(DATA_DIR, 'test_vol_h5')

# --- zip 압축 해제 (최초 1회, 이후 캐시 사용) ---
_train_cached = len(glob.glob(os.path.join(TRAIN_NPZ_DIR, '*.npz'))) >= 5
_test_cached  = len(glob.glob(os.path.join(TEST_H5_DIR,   '*.npy.h5'))) >= 1

if not (_train_cached and _test_cached):
    if os.path.exists(GDRIVE_ZIP):
        import zipfile
        print(f'Drive에서 압축 해제 중: {GDRIVE_ZIP}')
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(GDRIVE_ZIP, 'r') as zf:
            zf.extractall(DATA_DIR)
        # zip 내부 폴더가 synapse/ 인 경우 한 단계 올리기
        _inner = os.path.join(DATA_DIR, 'synapse')
        if os.path.exists(_inner):
            import shutil
            for item in os.listdir(_inner):
                shutil.move(os.path.join(_inner, item), DATA_DIR)
            os.rmdir(_inner)
        print(f'압축 해제 완료: train_npz {len(glob.glob(os.path.join(TRAIN_NPZ_DIR,"*.npz")))}개  '
              f'| test_vol_h5 {len(glob.glob(os.path.join(TEST_H5_DIR,"*.npy.h5")))}개')
    else:
        print('[데이터 없음] 아래 안내를 따라 데이터를 준비하세요:')
        print('  1. 다음 링크에서 데이터 다운로드:')
        print('     https://drive.google.com/drive/folders/1ACJEoTp-uqfFJ73qS3eUObQh52nGuzCd')
        print('  2. 로컬에서 zip으로 압축:')
        print('     zip -r synapse.zip synapse/')
        print('  3. Google Drive에 업로드:')
        print('     MyDrive/imbalanced-data-LWCE/synapse/synapse.zip')
        print('  4. 이 셀 재실행')
else:
    print(f'캐시 사용: train_npz {len(glob.glob(os.path.join(TRAIN_NPZ_DIR,"*.npz")))}개  '
          f'| test_vol_h5 {len(glob.glob(os.path.join(TEST_H5_DIR,"*.npy.h5")))}개')

train_files = sorted(glob.glob(os.path.join(TRAIN_NPZ_DIR, '*.npz')))
test_files  = sorted(glob.glob(os.path.join(TEST_H5_DIR,   '*.npy.h5')))
print(f'Train slices: {len(train_files)}  |  Test volumes: {len(test_files)}')
assert len(train_files) > 0, 'Train 데이터 없음. 위 안내를 따라 데이터를 배치하세요.'

# --- 클래스 정의 ---
NUM_CLASSES = 9
CLASS_NAMES = ['background', 'aorta', 'gallbladder', 'spleen',
               'left_kidney', 'right_kidney', 'liver', 'stomach', 'pancreas']

# --- Dataset 클래스 ---
class SynapseDataset(Dataset):
    def __init__(self, npz_files, img_size=224, augment=False):
        self.files    = npz_files
        self.img_size = img_size
        self.augment  = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.float32)
        label = data['label'].astype(np.int64)

        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        label = cv2.resize(label, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image = (image - image.min()) / (image.max() - image.min() + 1e-8)
        image = np.stack([image] * 3, axis=0).astype(np.float32)

        if self.augment:
            img_hw = image.transpose(1, 2, 0)
            if random.random() > 0.5: img_hw = np.fliplr(img_hw).copy(); label = np.fliplr(label).copy()
            if random.random() > 0.5: img_hw = np.flipud(img_hw).copy(); label = np.flipud(label).copy()
            image = img_hw.transpose(2, 0, 1)

        return torch.from_numpy(image), torch.from_numpy(label).long()

# --- DataLoader ---
tr_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)
train_loader = DataLoader(SynapseDataset(tr_files,  augment=True),  batch_size=12, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(SynapseDataset(val_files, augment=False), batch_size=12, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(tr_files)} slices  |  Val: {len(val_files)} slices')

# --- 클래스 비율 계산 ---
print('클래스 비율 계산 중...')
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files, desc='Counting'):
    lbl = np.load(fp)['label'].astype(np.int64)
    for c in range(NUM_CLASSES):
        class_counts[c] += int((lbl == c).sum())
class_counts = class_counts.tolist()
print('\nClass pixel counts:')
for i, (n, c) in enumerate(zip(CLASS_NAMES, class_counts)):
    ratio = class_counts[0] / (c + 1)
    print(f'  [{i}] {n:<15}: {c:>12,}  (BG:FG = {ratio:.0f}:1)')


캐시 사용: train_npz 2211개  | test_vol_h5 12개
Train slices: 2211  |  Test volumes: 12
Train: 1989 slices  |  Val: 222 slices
클래스 비율 계산 중...


Counting: 100%|██████████| 1989/1989 [00:09<00:00, 201.46it/s]


Class pixel counts:
  [0] background     :  498,855,150  (BG:FG = 1:1)
  [1] aorta          :      833,775  (BG:FG = 598:1)
  [2] gallbladder    :      216,180  (BG:FG = 2308:1)
  [3] spleen         :    1,217,254  (BG:FG = 410:1)
  [4] left_kidney    :    1,213,135  (BG:FG = 411:1)
  [5] right_kidney   :   12,606,515  (BG:FG = 40:1)
  [6] liver          :      577,023  (BG:FG = 865:1)
  [7] stomach        :    2,771,371  (BG:FG = 180:1)
  [8] pancreas       :    3,114,013  (BG:FG = 160:1)


In [12]:
# === Cell 2: TransUNet 모델 정의 ===
# --- TransUNet 모델 로드 ---
from networks.vit_seg_modeling import VisionTransformer as ViT_seg
from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

config_vit = CONFIGS_ViT_seg['R50-ViT-B_16']
config_vit.n_classes   = NUM_CLASSES
config_vit.n_skip       = 3
config_vit.patches.grid = (14, 14)   # 224 / 16

def build_transunet():
    model = ViT_seg(config_vit, img_size=224, num_classes=NUM_CLASSES)
    model.load_from(weights=np.load(VIT_WEIGHTS))
    return model.to(device)

# --- 검증 함수 ---
def compute_val_metrics(model, loader):
    """Val Dice (클래스별 + mDice) 계산"""
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)  # background 제외
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            preds  = torch.argmax(logits, dim=1)

            for c_idx, c in enumerate(range(1, NUM_CLASSES)):  # fg classes only
                p = (preds  == c).float()
                t = (masks  == c).float()
                inter = (p * t).sum()
                union = p.sum() + t.sum()
                if union > 0:
                    dice_per_class[c_idx] += (2. * inter / (union + 1e-8)).item()

            n_batches += 1

    dice_per_class /= n_batches
    return dice_per_class, float(np.mean(dice_per_class))

print("TransUNet 모델 준비 완료 (build_transunet() 로 인스턴스 생성)")
print(f"클래스 수: {NUM_CLASSES}  |  Foreground classes: {CLASS_NAMES[1:]}")


TransUNet 모델 준비 완료 (build_transunet() 로 인스턴스 생성)
클래스 수: 9  |  Foreground classes: ['aorta', 'gallbladder', 'spleen', 'left_kidney', 'right_kidney', 'liver', 'stomach', 'pancreas']


In [13]:
# === Cell 3: 학습 함수 ===
def train_transunet(loss_name, alpha=1.0, gamma=2.0, epochs=30, lr=1e-4,
                    subset_ratio=1.0, tag=""):
    """
    loss_name    : 'ce_dice' | 'plwce_dice' | 'pwce_dice' | 'plwce_focal_dice'
    alpha        : PLWCE / PWCE의 alpha 파라미터
    subset_ratio : Optuna 탐색용 축소 비율 (0~1). 1.0이면 전체 데이터
    tag          : 식별 태그 (저장 파일명용)
    """
    model     = build_transunet()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    name = f"{loss_name}_alpha{alpha:.2f}" if alpha != 1.0 else loss_name
    if tag: name = f"{tag}_{name}"
    print(f"\n{'='*60}\n{name}  (epochs={epochs}, subset={subset_ratio:.0%})\n{'='*60}")

    # subset_ratio 적용 (Optuna 탐색 시 빠른 평가용)
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds    = torch.utils.data.Subset(train_loader.dataset,
                                            random.sample(range(len(train_loader.dataset)), n))
        sub_loader = DataLoader(sub_ds, batch_size=12, shuffle=True, num_workers=2)
    else:
        sub_loader = train_loader

    best_dice = 0.0
    history   = {'loss': [], 'val_mdice': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(sub_loader, desc=f"Ep{epoch+1:02d}/{epochs}", leave=False, disable=os.environ.get("TQDM_DISABLE") == "1"):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()

            logits = model(imgs)                    # (B, 9, H, W)
            loss   = criterion(logits, masks)       # CE 교체 부분
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(sub_loader)
        _, mdice = compute_val_metrics(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_mdice'].append(mdice)

        print(f"Ep{epoch+1:02d} | Loss:{avg_loss:.4f} | Val mDice:{mdice:.4f}", end="")
        if mdice > best_dice:
            best_dice = mdice
            torch.save(model.state_dict(), f'/tmp/best_transunet_{name}.pth')
            print("  <- Best!", end="")
        print()

    model.load_state_dict(torch.load(f'/tmp/best_transunet_{name}.pth', weights_only=True))
    print(f"최고 Val mDice: {best_dice:.4f}")
    return model, history, best_dice

print("train_transunet() 함수 정의 완료")


train_transunet() 함수 정의 완료


In [14]:
# === Cell 4: Optuna alpha/gamma 탐색 ===
import traceback as _tb
os.environ['TQDM_DISABLE'] = '1'

# --- Optuna alpha 탐색 ---
# proxy 설정: subset_ratio=0.15, epochs=5 로 빠른 평가 후 최적 alpha 결정

ALPHA_LOW  = 2.5
ALPHA_HIGH = 15.0
PROXY_EPOCHS     = 5     # 탐색용 빠른 훈련
PROXY_SUBSET     = 0.15  # 전체 슬라이스의 15%만 사용
N_TRIALS         = 30    # 탐색 횟수
N_TRIALS_PF      = N_TRIALS * 2  # PLWCE+Focal: 파라미터 2개(alpha,gamma)이므로 2배

def make_alpha_objective(loss_name):
    """loss_name 별 Optuna objective 생성"""
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, mdice = train_transunet(
                loss_name   = loss_name,
                alpha       = alpha,
                epochs      = PROXY_EPOCHS,
                subset_ratio= PROXY_SUBSET,
                tag         = f"trial{trial.number}"
            )
            return mdice
        except Exception as e:
            print(f"Trial {trial.number} 실패:"); _tb.print_exc()
            return 0.0
    return objective

# --- PLWCE alpha 탐색 ---
print(f"\n{'='*60}")
print(f"[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH},  {N_TRIALS} trials)")
print(f"{'='*60}")

study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'transunet_plwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_plwce.optimize(make_alpha_objective('plwce_dice'),
                     n_trials=N_TRIALS, show_progress_bar=True)

best_alpha_plwce = study_plwce.best_params['alpha']
print(f"\n[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val mDice = {study_plwce.best_value:.4f})")

# --- PWCE alpha 탐색 ---
print(f"\n{'='*60}")
print(f"[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH},  {N_TRIALS} trials)")
print(f"{'='*60}")

study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'transunet_pwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pwce.optimize(make_alpha_objective('pwce_dice'),
                    n_trials=N_TRIALS, show_progress_bar=True)

best_alpha_pwce = study_pwce.best_params['alpha']
print(f"\n[PWCE] 최적 alpha = {best_alpha_pwce:.4f}  (Val mDice = {study_pwce.best_value:.4f})")

# --- PLWCE+Focal joint alpha+gamma 탐색 ---
GAMMA_LOW  = 0.5
GAMMA_HIGH = 5.0

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
    gamma = trial.suggest_float('gamma', GAMMA_LOW, GAMMA_HIGH)
    try:
        _, _, mdice = train_transunet(
            loss_name    = 'plwce_focal_dice',
            alpha        = alpha,
            gamma        = gamma,
            epochs       = PROXY_EPOCHS,
            subset_ratio = PROXY_SUBSET,
            tag          = f"trial{trial.number}"
        )
        return mdice
    except Exception as e:
        print(f"Trial {trial.number} 실패:"); _tb.print_exc()
        return 0.0

print(f"\n{'='*60}")
print(f"[Optuna] PLWCE+Focal joint alpha+gamma 탐색  ({N_TRIALS_PF} trials)")
print(f"{'='*60}")

study_pf = optuna.create_study(
    direction  = 'maximize',
    study_name = 'transunet_plwce_focal',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF, show_progress_bar=True)

best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f"\n[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  (Val mDice={study_pf.best_value:.4f})")


# --- Optuna 결과 저장 ---
optuna_results = {
    'plwce': {'best_alpha': best_alpha_plwce, 'best_proxy_mdice': study_plwce.best_value,
              'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                          'value': t.value} for t in study_plwce.trials if t.value is not None]},
    'pwce':  {'best_alpha': best_alpha_pwce,  'best_proxy_mdice': study_pwce.best_value,
              'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                          'value': t.value} for t in study_pwce.trials if t.value is not None]},
    'plwce_focal': {'best_alpha': best_alpha_pf, 'best_gamma': best_gamma_pf,
                    'best_proxy_mdice': study_pf.best_value,
                    'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                                'gamma': t.params.get('gamma'),
                                'value': t.value} for t in study_pf.trials if t.value is not None]},
}
with open(os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f"\nOptuna 결과 저장: {os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json')}")

# --- Optuna 탐색 곡선 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, name in [(axes[0], study_plwce, 'PLWCE'), (axes[1], study_pwce, 'PWCE')]:
    trials   = [t for t in study.trials if t.value is not None]
    alphas   = [t.params['alpha'] for t in trials]
    values   = [t.value for t in trials]
    best_a   = study.best_params['alpha']
    best_v   = study.best_value

    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best α={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val mDice (proxy)')
    ax.set_title(f'{name} alpha 탐색 결과'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_optuna_search.png'), dpi=100)
plt.show()
print(f"탐색 결과 이미지 저장: {os.path.join(RESULTS_DIR, 'pancreas_optuna_search.png')}")

# PLWCE+Focal 2D scatter 시각화
fig, ax = plt.subplots(figsize=(7, 5))
trials_pf = [t for t in study_pf.trials if t.value is not None]
alphas_pf  = [t.params['alpha'] for t in trials_pf]
gammas_pf  = [t.params['gamma'] for t in trials_pf]
values_pf  = [t.value           for t in trials_pf]
sc = ax.scatter(alphas_pf, gammas_pf, c=values_pf, cmap='viridis', s=60, alpha=0.8)
ax.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, marker='*', zorder=5,
           label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax, label='Val mDice')
ax.set_xlabel('alpha'); ax.set_ylabel('gamma')
ax.set_title('PLWCE+Focal Optuna 탐색 (Synapse)'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'Synapse_optuna_search_pf.png'), dpi=100)
plt.show()
print(f"탐색 결과 이미지 저장: {os.path.join(RESULTS_DIR, 'Synapse_optuna_search_pf.png')}")

os.environ.pop('TQDM_DISABLE', None)



[Optuna] PLWCE alpha 탐색  (범위: 2.5~15.0,  30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]

[plwce_dice] Weights (plwce): Generated.

trial0_plwce_dice_alpha13.46  (epochs=5, subset=15%)
Ep01 | Loss:1.5436 | Val mDice:0.0822  <- Best!
Ep02 | Loss:1.3931 | Val mDice:0.0968  <- Best!
Ep03 | Loss:1.2447 | Val mDice:0.1726  <- Best!
Ep04 | Loss:1.1583 | Val mDice:0.1883  <- Best!
Ep05 | Loss:1.0847 | Val mDice:0.2153  <- Best!
최고 Val mDice: 0.2153
[plwce_dice] Weights (plwce): Generated.

trial1_plwce_dice_alpha12.11  (epochs=5, subset=15%)
Ep01 | Loss:1.4968 | Val mDice:0.0337  <- Best!
Ep02 | Loss:1.3023 | Val mDice:0.0745  <- Best!
Ep03 | Loss:1.1921 | Val mDice:0.1066  <- Best!
Ep04 | Loss:1.1382 | Val mDice:0.1633  <- Best!
Ep05 | Loss:1.1098 | Val mDice:0.1831  <- Best!
최고 Val mDice: 0.1831
[plwce_dice] Weights (plwce): Generated.

trial2_plwce_dice_alpha13.99  (epochs=5, subset=15%)
Ep01 | Loss:1.4944 | Val mDice:0.0894  <- Best!
Ep02 | Loss:1.2796 | Val mDice:0.0598
Ep03 | Loss:1.1339 | Val mDice:0.2001  <- Best!
Ep04 | Loss:1.0553 | Val mDice:0.1430
Ep05 | Loss:1.0115 | 

  0%|          | 0/30 [00:00<?, ?it/s]

[pwce_dice] Weights (pwce): Generated.

trial0_pwce_dice_alpha10.38  (epochs=5, subset=15%)
Ep01 | Loss:1.3709 | Val mDice:0.0030  <- Best!
Ep02 | Loss:1.1733 | Val mDice:0.0075  <- Best!
Ep03 | Loss:1.0033 | Val mDice:0.0044
Ep04 | Loss:0.7177 | Val mDice:0.0115  <- Best!
Ep05 | Loss:0.6556 | Val mDice:0.0098
최고 Val mDice: 0.0115
[pwce_dice] Weights (pwce): Generated.

trial1_pwce_dice_alpha14.54  (epochs=5, subset=15%)
Ep01 | Loss:nan | Val mDice:0.0000
Ep02 | Loss:nan | Val mDice:0.0000
Ep03 | Loss:nan | Val mDice:0.0000
Ep04 | Loss:nan | Val mDice:0.0000
Ep05 | Loss:nan | Val mDice:0.0000
Trial 1 실패:


...

Traceback (most recent call last):
  File "/tmp/ipykernel_537030/2741198833.py", line 19, in objective
    _, _, mdice = train_transunet(
                  ^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_537030/4006397291.py", line 58, in train_transunet
    model.load_state_dict(torch.load(f'/tmp/best_transunet_{name}.pth', weights_only=True))
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundErro

[pwce_dice] Weights (pwce): Generated.

trial9_pwce_dice_alpha5.78  (epochs=5, subset=15%)
Ep01 | Loss:1.3404 | Val mDice:0.0206  <- Best!
Ep02 | Loss:0.9804 | Val mDice:0.0054
Ep03 | Loss:0.8225 | Val mDice:0.0025
Ep04 | Loss:0.6917 | Val mDice:0.0139
Ep05 | Loss:0.6449 | Val mDice:0.0133
최고 Val mDice: 0.0206
[pwce_dice] Weights (pwce): Generated.

trial10_pwce_dice_alpha4.67  (epochs=5, subset=15%)
Ep01 | Loss:1.3553 | Val mDice:0.0083  <- Best!
Ep02 | Loss:1.0558 | Val mDice:0.0013
Ep03 | Loss:0.8651 | Val mDice:0.0039
Ep04 | Loss:0.8185 | Val mDice:0.0044
Ep05 | Loss:0.7475 | Val mDice:0.0039
최고 Val mDice: 0.0083
[pwce_dice] Weights (pwce): Generated.

trial11_pwce_dice_alpha2.51  (epochs=5, subset=15%)
Ep01 | Loss:1.5248 | Val mDice:0.0050  <- Best!
Ep02 | Loss:1.2513 | Val mDice:0.0071  <- Best!
Ep03 | Loss:1.0176 | Val mDice:0.0063
Ep04 | Loss:0.9126 | Val mDice:0.0096  <- Best!
Ep05 | Loss:0.8562 | Val mDice:0.0071
최고 Val mDice: 0.0096
[pwce_dice] Weights (pwce): Generated.

tr

  0%|          | 0/60 [00:00<?, ?it/s]

[plwce_focal_dice] Weights (plwce): Generated.

trial0_plwce_focal_dice_alpha13.85  (epochs=5, subset=15%)
Ep01 | Loss:0.5102 | Val mDice:0.0929  <- Best!
Ep02 | Loss:0.4997 | Val mDice:0.0816
Ep03 | Loss:0.4935 | Val mDice:0.1023  <- Best!
Ep04 | Loss:0.4864 | Val mDice:0.1262  <- Best!
Ep05 | Loss:0.4829 | Val mDice:0.1388  <- Best!
최고 Val mDice: 0.1388
[plwce_focal_dice] Weights (plwce): Generated.

trial1_plwce_focal_dice_alpha5.00  (epochs=5, subset=15%)
Ep01 | Loss:0.6778 | Val mDice:0.0969  <- Best!
Ep02 | Loss:0.6370 | Val mDice:0.1159  <- Best!
Ep03 | Loss:0.6203 | Val mDice:0.0982
Ep04 | Loss:0.6068 | Val mDice:0.1725  <- Best!
Ep05 | Loss:0.5991 | Val mDice:0.1718
최고 Val mDice: 0.1725
[plwce_focal_dice] Weights (plwce): Generated.

trial2_plwce_focal_dice_alpha12.24  (epochs=5, subset=15%)
Ep01 | Loss:0.5098 | Val mDice:0.0953  <- Best!
Ep02 | Loss:0.4951 | Val mDice:0.1067  <- Best!
Ep03 | Loss:0.4881 | Val mDice:0.1357  <- Best!
Ep04 | Loss:0.4835 | Val mDice:0.1493  <- Be

'1'

In [15]:
# === Cell 5: 전체 Loss 비교 학습 ===
# --- 최적 alpha 불러오기 (이전 셀이 실행됐다면 메모리에 있음, 없으면 JSON에서 로드) ---
try:
    _ = best_alpha_plwce
except NameError:
    with open(os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json')) as f:
        d = json.load(f)
    best_alpha_plwce = d['plwce']['best_alpha']
    best_alpha_pwce  = d['pwce']['best_alpha']
    best_alpha_pf    = d['plwce_focal']['best_alpha']
    best_gamma_pf    = d['plwce_focal']['best_gamma']

FINAL_EPOCHS = 150   # 최종 비교 학습 epoch 수
FINAL_LR     = 1e-4

# --- 3가지 Loss 최종 학습 ---
experiments = [
    ('ce_dice',          1.0,              2.0,            'CE+Dice (기준선)'),
    ('plwce_dice',       best_alpha_plwce, 2.0,            f'PLWCE+Dice (alpha={best_alpha_plwce:.2f})'),
    ('pwce_dice',        best_alpha_pwce,  2.0,            f'PWCE+Dice  (alpha={best_alpha_pwce:.2f})'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf,  f'PLWCE+Focal+Dice (alpha={best_alpha_pf:.2f}, gamma={best_gamma_pf:.2f})'),
]

final_results = {}
for loss_name, alpha, gamma, label in experiments:
    print(f"\n>>> {label}")
    model, history, best_dice = train_transunet(
        loss_name = loss_name,
        alpha     = alpha,
        gamma     = gamma,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        subset_ratio = 1.0,
        tag       = 'final'
    )
    final_results[label] = {'model': model, 'history': history, 'best_dice': best_dice,
                            'loss_name': loss_name, 'alpha': alpha, 'gamma': gamma}

print("\n\n[최종 학습 결과 요약]")
print(f"{'실험':<35} {'Best Val mDice':>14}")
print("-" * 51)
for label, r in final_results.items():
    print(f"{label:<35} {r['best_dice']:>14.4f}")



>>> CE+Dice (기준선)

final_ce_dice  (epochs=150, subset=100%)


Ep01 | Loss:1.0219 | Val mDice:0.1074  <- Best!


Ep02 | Loss:0.6239 | Val mDice:0.1149  <- Best!


Ep03 | Loss:0.5148 | Val mDice:0.2110  <- Best!


Ep04 | Loss:0.4719 | Val mDice:0.2071


...

Ep146 | Loss:0.0551 | Val mDice:0.9035


Ep147 | Loss:0.0561 | Val mDice:0.9038


Ep148 | Loss:0.0547 | Val mDice:0.9033


Ep149 | Loss:0.0548 | Val mDice:0.9035


Ep150 | Loss:0.0603 | Val mDice:0.9035
최고 Val mDice: 0.9042


[최종 학습 결과 요약]
실험                                  Best Val mDice
---------------------------------------------------
CE+Dice (기준선)                               0.9080
PLWCE+Dice (alpha=6.44)                     0.9097
PWCE+Dice  (alpha=3.98)                     0.7671
PLWCE+Focal+Dice (alpha=8.37, gamma=4.76)         0.9042


In [16]:
# === Cell 6: 평가 및 결과 저장 (클래스별 Dice + Excel) ===
# --- Test set 평가 (클래스별 Dice) ---
def evaluate_on_testset(model):
    """Test volume(.h5)에 대해 클래스별 Dice 계산"""
    model.eval()
    dice_all = np.zeros(NUM_CLASSES - 1)  # fg only
    n_cases  = 0

    for h5_path in tqdm(test_files, desc="Test eval"):
        with h5py.File(h5_path, 'r') as f:
            vol   = f['image'][:]   # (D, H, W) or (H, W, D)
            label = f['label'][:]

        # 슬라이스 단위로 예측
        preds = np.zeros_like(label)
        for s in range(vol.shape[0]):
            sl = vol[s].astype(np.float32)
            sl = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
            sl = cv2.resize(sl, (224, 224), interpolation=cv2.INTER_LINEAR)
            sl = torch.from_numpy(np.stack([sl]*3)).unsqueeze(0).float().to(device)
            with torch.no_grad():
                logit = model(sl)
                pred  = torch.argmax(logit, dim=1).squeeze().cpu().numpy()
            pred_orig = cv2.resize(pred.astype(np.float32),
                                   (label.shape[2], label.shape[1]),
                                   interpolation=cv2.INTER_NEAREST).astype(np.int64)
            preds[s] = pred_orig

        for c_idx, c in enumerate(range(1, NUM_CLASSES)):
            p = (preds  == c).astype(float)
            t = (label  == c).astype(float)
            inter = (p * t).sum()
            union = p.sum() + t.sum()
            if union > 0:
                dice_all[c_idx] += 2. * inter / (union + 1e-8)
        n_cases += 1

    return dice_all / n_cases

# --- 모든 모델 Test 평가 ---
test_scores = {}
if test_files:
    for label, r in final_results.items():
        print(f"\n{label} 테스트 평가 중...")
        per_class_dice = evaluate_on_testset(r['model'])
        mdice = float(np.mean(per_class_dice))
        test_scores[label] = {'per_class': per_class_dice.tolist(), 'mDice': mdice}
        print(f"  mDice = {mdice:.4f}")
        for cn, dc in zip(CLASS_NAMES[1:], per_class_dice):
            print(f"    {cn:<15}: {dc:.4f}")
else:
    print("테스트 데이터 없음 — Val 결과로 대체")
    for label, r in final_results.items():
        dice_pc, mdice = compute_val_metrics(r['model'], val_loader)
        test_scores[label] = {'per_class': dice_pc.tolist(), 'mDice': mdice}

# --- 클래스별 Dice 비교 바차트 ---
fg_names = CLASS_NAMES[1:]
x = np.arange(len(fg_names))
width = 0.25
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']
for i, (label, scores) in enumerate(test_scores.items()):
    ax.bar(x + i * width, scores['per_class'], width,
           label=f"{label}  (mDice={scores['mDice']:.4f})", color=colors[i], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(fg_names, rotation=30, ha='right')
ax.set_ylabel('Dice Score'); ax.set_title('TransUNet — 클래스별 Dice 비교 (CE vs PLWCE vs PWCE)')
ax.legend(); ax.grid(axis='y', alpha=0.5); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_class_dice.png'), dpi=100)
plt.show()

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, r) in enumerate(final_results.items()):
    ax1.plot(r['history']['loss'],     label=label, color=colors[i])
    ax2.plot(r['history']['val_mdice'],label=label, color=colors[i])
ax1.set_title('Train Loss');   ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.set_title('Val mDice');    ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_training_curves.png'), dpi=100)
plt.show()

# --- 최종 결과 저장 ---
save = {}
for label, scores in test_scores.items():
    r = final_results[label]
    save[label] = {
        'loss_name'   : r['loss_name'],
        'alpha'       : r['alpha'],
        'gamma'       : r['gamma'],
        'best_val_mdice': r['best_dice'],
        'test_mDice'  : scores['mDice'],
        'per_class_dice': {cn: float(d) for cn, d in zip(fg_names, scores['per_class'])},
        'best_alpha_optuna': {
            'plwce': best_alpha_plwce,
            'pwce' : best_alpha_pwce,
            'plwce_focal': {'alpha': best_alpha_pf, 'gamma': best_gamma_pf},
        }
    }

with open(os.path.join(RESULTS_DIR, 'pancreas_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save, f, indent=2, ensure_ascii=False)
print(f"JSON 저장: {os.path.join(RESULTS_DIR, 'pancreas_final_results.json')}")

# --- Excel 저장 ---
import pandas as pd
rows = []
for label, s in save.items():
    row = {'Loss': label, 'Test mDice': round(s['test_mDice'], 4),
           'Best Val mDice': round(s['best_val_mdice'], 4),
           'alpha': round(s['alpha'], 4), 'gamma': round(s['gamma'], 4)}
    row.update({cn: round(d, 4) for cn, d in s['per_class_dice'].items()})
    rows.append(row)
df = pd.DataFrame(rows)
_xlsx = os.path.join(RESULTS_DIR, 'pancreas_final_results.xlsx')
df.to_excel(_xlsx, index=False)
print(f"Excel 저장: {_xlsx}")

print("\n[최종 결과 요약]")
print(f"{'실험':<40} {'Test mDice':>10}")
print("-" * 52)
for label, s in save.items():
    print(f"{label:<40} {s['test_mDice']:>10.4f}")



CE+Dice (기준선) 테스트 평가 중...


Test eval: 100%|██████████| 12/12 [01:27<00:00,  7.28s/it]


  mDice = 0.7589
    aorta          : 0.8261
    gallbladder    : 0.4850
    spleen         : 0.7931
    left_kidney    : 0.7697
    right_kidney   : 0.9402
    liver          : 0.5858
    stomach        : 0.8832
    pancreas       : 0.7880

PLWCE+Dice (alpha=6.44) 테스트 평가 중...


Test eval: 100%|██████████| 12/12 [01:19<00:00,  6.59s/it]


  mDice = 0.7619
    aorta          : 0.8264
    gallbladder    : 0.4690
    spleen         : 0.8077
    left_kidney    : 0.7662
    right_kidney   : 0.9366
    liver          : 0.5769
    stomach        : 0.8910
    pancreas       : 0.8215

PWCE+Dice  (alpha=3.98) 테스트 평가 중...


Test eval: 100%|██████████| 12/12 [01:19<00:00,  6.62s/it]


  mDice = 0.6510
    aorta          : 0.8080
    gallbladder    : 0.0009
    spleen         : 0.7773
    left_kidney    : 0.7177
    right_kidney   : 0.9079
    liver          : 0.5000
    stomach        : 0.8162
    pancreas       : 0.6798

PLWCE+Focal+Dice (alpha=8.37, gamma=4.76) 테스트 평가 중...


Test eval: 100%|██████████| 12/12 [01:19<00:00,  6.59s/it]


  mDice = 0.7426
    aorta          : 0.8107
    gallbladder    : 0.4656
    spleen         : 0.7767
    left_kidney    : 0.7448
    right_kidney   : 0.9410
    liver          : 0.5576
    stomach        : 0.8558
    pancreas       : 0.7886
JSON 저장: /root/imbalanced-data-LWCE/medical_data/results/Pancreas_MultiOrgan_CT/pancreas_final_results.json
Excel 저장: /root/imbalanced-data-LWCE/medical_data/results/Pancreas_MultiOrgan_CT/pancreas_final_results.xlsx

[최종 결과 요약]
실험                                       Test mDice
----------------------------------------------------
CE+Dice (기준선)                                0.7589
PLWCE+Dice (alpha=6.44)                      0.7619
PWCE+Dice  (alpha=3.98)                      0.6510
PLWCE+Focal+Dice (alpha=8.37, gamma=4.76)     0.7426
